# Testing batdetect2 pipeline features:

## **Working on features relevant to the use of this pipeline on recovered Audiomoth .wav recordings**

## 1) Figuring out our imports:

### a) Below are the imports pertaining to accessing data and metadata

In [1]:
from pathlib import Path
import glob
import exiftool
import suncalc
import soundfile as sf
import re

### b) Below are the imports pertaining to data manipulation

In [2]:
import numpy as np
import pandas as pd
import dask.dataframe as dd

### c) Below are the imports pertaining to data visualization

In [3]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors
import datetime as dt

### d) Below are the imports pertaining to the use of the MSDS pipeline

In [4]:
import sys

# append the path of the
# parent directory
sys.path.append('..')
sys.path.append('../src/')
sys.path.append('../src/models/bat_call_detector/batdetect2/')

import src.batdt2_pipeline as batdetect2_pipeline
import src.file_dealer as file_dealer
from cfg import get_config

In [5]:
SEATTLE_LATITUDE = 47.655181
SEATTLE_LONGITUDE = -122.293123

In [6]:
field_records = dd.read_csv(f'../field_records/ubna_2026.csv', dtype=str).compute()
field_records

,Date and Time deployed (local),Date and Time recovered (local),AudioMoth #,SD card #,SD card size (GB),Amount of data recovered (GB),Site,Latitude,Longitude,Firmware,...,Amplitude threshold,ON (secs),OFF (secs),Battery start (V),Battery end (V),Deployer,Scribe,Uploader,Upload folder name,Notes
0,2025-12-26T12:09:00,2026-01-08T13:56:00,O,032,256,216.9,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.556,4.552,AM,AK,AK,recover-20260108,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
1,2025-12-26T12:09:00,2026-01-08T13:46:00,A,027,256,216.8,Foliage,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.569,4.556,AM,AK,AK,recover-20260108,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
2,2025-12-26T12:09:00,2026-01-08T13:11:00,N,028,256,203.4,E18 Bridge,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.572,4.152,AM,AK,AK,recover-20260108,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
3,2025-12-26T12:18:00,2026-01-02T13:14:00,B,029,256,116.7,Central Pond,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.318,3.782,AM,AK,AK,recover-20260102,9 Ikea Batteries 0:00-24:00 UTC; silica gel p...
4,2025-12-26T12:09:00,2026-01-02T13:04:00,H,022,256,116.8,Carp Pond,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.036,3.761,AM,AK,AK,recover-20260102,9 Ikea Batteries 0:00-24:00 UTC; silica gel p...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,2026-08-17T13:23:00,(DATE-TIME),H,027,256,(AMOUNT-RECOVERED),Foliage,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.1,...,None,300,300,4.148,(VOLT-END),AK,AM,(UPLOADER),(RECOVER-DATE),9 Ikea Batteries 0:00-24:00 UTC; silica gel p...
111,2026-08-17T13:23:00,(DATE-TIME),J,028,256,(AMOUNT-RECOVERED),Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.1,...,None,300,300,4.189,(VOLT-END),AK,AM,(UPLOADER),(RECOVER-DATE),9 Ikea Batteries 0:00-24:00 UTC; silica gel p...
112,2026-08-17T13:23:00,(DATE-TIME),L,030,256,(AMOUNT-RECOVERED),Central Pond,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.1,...,None,300,300,4.147,(VOLT-END),AK,AM,(UPLOADER),(RECOVER-DATE),9 Ikea Batteries 0:00-24:00 UTC; silica gel p...
113,2026-08-17T13:23:00,(DATE-TIME),R,031,256,(AMOUNT-RECOVERED),E18 Bridge,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.1,...,None,300,300,4.358,(VOLT-END),AK,AM,(UPLOADER),(RECOVER-DATE),9 Ikea Batteries 0:00-24:00 UTC; silica gel p...


In [7]:
field_records.columns

Index([' Date and Time deployed (local) ', ' Date and Time recovered (local) ',
       ' AudioMoth # ', ' SD card # ', ' SD card size (GB) ',
       ' Amount of data recovered (GB) ', ' Site', ' Latitude ', ' Longitude ',
       ' Firmware ', ' GPS Module? ', ' Sampling rate (Hz) ', ' Gain ',
       ' Filter ', ' Amplitude threshold ', ' ON (secs) ', ' OFF (secs) ',
       ' Battery start (V) ', ' Battery end (V) ', ' Deployer ', ' Scribe ',
       ' Uploader ', ' Upload folder name ', ' Notes '],
      dtype='object')

In [8]:
tel_field_records = field_records[field_records[' Site']==' Telephone Field ']
tel_field_records = tel_field_records[-10:-1]
tel_field_records

,Date and Time deployed (local),Date and Time recovered (local),AudioMoth #,SD card #,SD card size (GB),Amount of data recovered (GB),Site,Latitude,Longitude,Firmware,...,Amplitude threshold,ON (secs),OFF (secs),Battery start (V),Battery end (V),Deployer,Scribe,Uploader,Upload folder name,Notes
68,2026-05-18T12:27:00,2026-06-01T16:42:00,023,028,256,231.4,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.564,2.920,AM,AK,ES,recover-20260601,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
74,2026-06-01T11:59:00,2026-06-10T13:55:00,J,027,256,150.7,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.123,3.781,AK,ES,AM,recover-20260610,9 Ikea Batteries 0:00-24:00 UTC; silica gel p...
80,2026-06-10T11:45:00,2026-06-24T12:18:00,023,033,256,232.6,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.11.0,...,None,300,300,4.562,4.564,AM,AK,AK,recover-20260624,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
82,2026-06-10T13:57:00,2026-06-17T14:30:00,046,STF_081,256,80.8,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-GPS-Sync-1.2.1,...,None,300,300,4.581,2.476,AM,AK,AK,recover-20260617,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
88,2026-06-24T10:45:00,2026-07-08T13:11:00,017,027,256,233.9,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.0,...,None,300,300,4.572,4.568,AM,VC,AM,recover-20260708,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
97,2026-07-08T13:15:00,2026-07-15T13:32:00,017,036,256,0.0,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.0,...,None,300,300,4.561,4.563,AM,VC,(UPLOADER),(RECOVER-DATE),3 Paleblue Batteries 0:00-24:00 UTC; silica g...
98,2026-07-15T11:53:00,2026-07-24T11:50:00,021,021,128,127.8,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.0,...,None,300,300,4.554,4.555,AM,VC,AK,recover-20260724,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
100,2026-07-24T10:23:00,2026-08-05T13:01:00,017,028,256,200.9,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.0,...,None,300,300,4.563,4.564,AM,AM,AK,recover-20260805,3 Paleblue Batteries 0:00-24:00 UTC; silica g...
107,2026-08-05T11:27:00,2026-08-17T14:48:00,023,033,256,1.0,Telephone Field,(LATITUDE),(LONGITUDE),AudioMoth-Firmware-Basic-1.12.0,...,None,300,300,4.560,4.563,AM,AK,AK,recover-20260817,3 Paleblue Batteries 0:00-24:00 UTC; silica g...


In [9]:
sites = ['E18 Bridge', 'Central Pond', 'Foliage', 'Telephone Field']

METRIC_TAGS = {'CALLRATE':'CR',
               'BOUTTIMEPERCENTAGE':'BTP',
               'ACTIVITYINDEX':'AI'}
COLNAME_TAGS = {'CALLRATE':'callrate',
               'BOUTTIMEPERCENTAGE':'bout_time_percentage',
               'ACTIVITYINDEX':'activity_index'}
PLOT_UPPER_LIM = {'CALLRATE':1e3,
               'BOUTTIMEPERCENTAGE':1e2,
               'ACTIVITYINDEX':1e2}

cfg = dict()
cfg['recover_folder'] = 'recover-20260102'
cfg['sd_unit'] = '029'
cfg['site'] = 'Central Pond'
cfg['output_dir'] = Path(f"../output_dir/{cfg['recover_folder']}")
cfg['cycle_length'] = 600
cfg['duration'] = 300
cfg["recording_start"] = '00:00'
cfg["recording_end"] = '23:59'

data_params = batdetect2_pipeline.get_params_relevant_to_data(cfg, data_drive_num=7)
cfg["csv_filename"] = f"bd2__{data_params['recover_folder']}_{data_params['audiomoth_folder']}"

data_params['resample_in_min'] = 30
data_params['resample_tag'] = f"{data_params['resample_in_min']}T"

data_params['detection_threshold_for_activity'] = 0.35
data_params['SNR_threshold_for_activity'] = 3

batdetect2_pipeline.construct_activity_arr(cfg, data_params)
for group in ['', 'LF', 'HF']:
    for cfg['METRIC'] in ['CALLRATE', 'BOUTTIMEPERCENTAGE', 'ACTIVITYINDEX']:
        cfg['METRIC_TAG'] = METRIC_TAGS[cfg['METRIC']]
        cfg['COL_TAG'] = COLNAME_TAGS[cfg['METRIC']]
        cfg['UPPER_LIM'] = PLOT_UPPER_LIM[cfg['METRIC']]
        activity_df = batdetect2_pipeline.shape_activity_array_into_grid(cfg, data_params, group)
        batdetect2_pipeline.plot_activity_grid(activity_df, cfg, data_params, group)

Searching for files from recover-20260102 and UBNA_029
Will save csv file to ../output_dir/recover-20260102/Central Pond
Error files exist!


### d) The below code was written to loop for all deployment sessions to have a single uniform format between deployment sessions. The `bd2__` detection files remain untouched.

In [ ]:
import re
from pathlib import Path

METRIC_TAGS = {
    'CALLRATE': 'CR',
    'BOUTTIMEPERCENTAGE': 'BTP',
    'ACTIVITYINDEX': 'AI',
}

COLNAME_TAGS = {
    'CALLRATE': 'callrate',
    'BOUTTIMEPERCENTAGE': 'bout_time_percentage',
    'ACTIVITYINDEX': 'activity_index',
}

PLOT_UPPER_LIM = {
    'CALLRATE': 1e3,
    'BOUTTIMEPERCENTAGE': 1e2,
    'ACTIVITYINDEX': 1e2,
}

output_root = Path("../output_dir")
first_recover = "recover-20260102"
last_recover = "recover-20260420"

recover_dirs = sorted(
    recover_dir
    for recover_dir in output_root.glob("recover-*")
    if (recover_dir.is_dir() and first_recover <= recover_dir.name <= last_recover)
)

for recover_dir in recover_dirs:

    # rglob handles bd2 files stored inside site subdirectories.
    for bd2_csv in sorted(recover_dir.rglob("bd2__*.csv")):

        # Matches both UBNA_029 and UBNA029 at the end of the filename.
        sd_match = re.search(r"UBNA_?(\d+)$", bd2_csv.stem)

        if sd_match is None:
            print(f"Skipping unrecognized filename: {bd2_csv.name}")
            continue

        cfg = {
            "recover_folder": recover_dir.name,
            "sd_unit": sd_match.group(1),  # Preserves leading zeros.
            "site": bd2_csv.parent.name,
            "output_dir": recover_dir,
            "cycle_length": 600,
            "duration": 300,
            "recording_start": "00:00",
            "recording_end": "23:59",
        }

        print(
            f"\nProcessing {cfg['recover_folder']}, "
            f"UBNA_{cfg['sd_unit']}: {bd2_csv.name}"
        )

        data_params = batdetect2_pipeline.get_params_relevant_to_data(cfg, data_drive_num=7)

        # Use the filename that was actually discovered.
        cfg["csv_filename"] = bd2_csv.stem

        # Ensure the functions read and write alongside that bd2 CSV.
        data_params["output_dir"] = bd2_csv.parent

        data_params["resample_in_min"] = 30
        data_params["resample_tag"] = f"{data_params['resample_in_min']}T"
        data_params["detection_threshold_for_activity"] = 0.35
        for snr in [3, 7]:
            data_params["SNR_threshold_for_activity"] = snr
            batdetect2_pipeline.construct_activity_arr(cfg, data_params)

            for group in ["", "LF", "HF"]:
                for metric in ["CALLRATE", "BOUTTIMEPERCENTAGE", "ACTIVITYINDEX"]:
                    cfg["METRIC"] = metric
                    cfg["METRIC_TAG"] = METRIC_TAGS[metric]
                    cfg["COL_TAG"] = COLNAME_TAGS[metric]
                    cfg["UPPER_LIM"] = PLOT_UPPER_LIM[metric]

                    activity_df = batdetect2_pipeline.shape_activity_array_into_grid(cfg, data_params, group,)
                    batdetect2_pipeline.plot_activity_grid(activity_df, cfg, data_params, group)

                    year = '2026'
                    data_params['selection_of_dates'] = f'recover-{year}*'
                    cumulative_activity_df = batdetect2_pipeline.construct_cumulative_activity(data_params, cfg, group)
                    data_params['show_PST'] = False
                    data_params['UPPER_LIM'] = cfg['UPPER_LIM']
                    data_params['METRIC_TAG'] = cfg['METRIC_TAG']
                    data_params['METRIC'] = cfg['METRIC']
                    batdetect2_pipeline.plot_cumulative_activity(cumulative_activity_df, data_params, group)


Processing recover-20260102, UBNA_022: bd2__recover-20260102_UBNA_022.csv
Searching for files from recover-20260102 and UBNA_022
Will save csv file to ../output_dir/recover-20260102/Carp Pond
Error files exist!

Processing recover-20260102, UBNA_029: bd2__recover-20260102_UBNA_029.csv
Searching for files from recover-20260102 and UBNA_029
Will save csv file to ../output_dir/recover-20260102/Central Pond
Error files exist!

Processing recover-20260108, UBNA_024: bd2__recover-20260108_UBNA_024.csv
Searching for files from recover-20260108 and UBNA_024
Will save csv file to ../output_dir/recover-20260108/Carp Pond
Error files exist!

Processing recover-20260108, UBNA_031: bd2__recover-20260108_UBNA_031.csv
Searching for files from recover-20260108 and UBNA_031
Will save csv file to ../output_dir/recover-20260108/Central Pond
Error files exist!

Processing recover-20260108, UBNA_028: bd2__recover-20260108_UBNA_028.csv
Searching for files from recover-20260108 and UBNA_028
Will save csv fi